# Stage 5: Results Summary Dashboard

This notebook aggregates and displays all evaluation and training results from the MoshiLite distillation pipeline.

## 1. Setup & Mount Drive

In [ ]:
import os
import json
import pandas as pd
from IPython.display import display
from google.colab import drive

# 1. Setup paths
drive.mount('/content/drive')
EXPERIMENT_ID = "prune30_v1"
BASE_EVAL_DIR = f"/content/drive/MyDrive/moshilite/eval/{EXPERIMENT_ID}"
kd_dir = os.path.join(BASE_EVAL_DIR, "kd")

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

## 2. Training Summary & Post-KD Results

In [ ]:
kd_records = []

if os.path.exists(kd_dir):
    for run_folder in sorted(os.listdir(kd_dir)):
        run_path = os.path.join(kd_dir, run_folder)
        if not os.path.isdir(run_path): continue
            
        record = {"Model Variant": run_folder}
        
        # 1. Load Training Summary
        summary_path = os.path.join(run_path, "training_summary.json")
        if os.path.exists(summary_path):
            with open(summary_path, 'r') as f:
                summary = json.load(f)
            record["Total steps"] = summary.get("total_steps", 0)
            record["Best val loss"] = f"{summary.get('best_val_loss', 0):.4f}"
            record["Final train loss"] = f"{summary.get('final_train_loss', 0):.4f}"
            record["Training time (min)"] = f"{summary.get('elapsed_seconds', 0) / 60.0:.1f}"
            
        # 2. Load Post-KD Mimicking Evaluation
        post_eval_path = os.path.join(run_path, "post_kd_eval.json")
        if os.path.exists(post_eval_path):
            with open(post_eval_path, 'r') as f:
                post_eval = json.load(f)
            
            if "post_kd" in post_eval:
                post_eval = post_eval["post_kd"]
                
            record["text_token_acc"] = f"{post_eval.get('text_token_acc', 0.0):.4f}"
            record["audio_cb0_token_acc"] = f"{post_eval.get('audio_cb0_token_acc', 0.0):.4f}"
            record["text_top5_agree"] = f"{post_eval.get('text_top5_agree', 0.0):.4f}"
            record["audio_cb0_top5_agree"] = f"{post_eval.get('audio_cb0_top5_agree', 0.0):.4f}"
            record["text_kl_div"] = f"{post_eval.get('text_kl_div', 0.0):.4f}"
            record["audio_cb0_kl_div"] = f"{post_eval.get('audio_cb0_kl_div', 0.0):.4f}"
            record["text_perplexity"] = f"{post_eval.get('text_perplexity', 0.0):.2f}"
            record["audio_cb0_perplexity"] = f"{post_eval.get('audio_cb0_perplexity', 0.0):.2f}"
            record["val_loss_l2"] = f"{post_eval.get('val_loss_l2', 0.0):.4f}"
            
        if len(record) > 1:
            kd_records.append(record)

    if kd_records:
        df_kd_summary = pd.DataFrame(kd_records).set_index("Model Variant")
        print("--- Training Summary & Post-KD Results ---")
        display(df_kd_summary)
else:
    print(f"Directory not found: {kd_dir}")

## 3. Pre-KD (Post-Prune) vs Post-KD Multi-Index Comparison Table

In [ ]:
compare_records = []
metrics_to_compare = [
    "text_token_acc", "audio_cb0_token_acc", 
    "text_top5_agree", "audio_cb0_top5_agree", 
    "text_kl_div", "audio_cb0_kl_div", 
    "text_perplexity", "audio_cb0_perplexity", 
    "val_loss_l2"
]

if os.path.exists(kd_dir):
    for run_folder in sorted(os.listdir(kd_dir)):
        run_path = os.path.join(kd_dir, run_folder)
        if not os.path.isdir(run_path): continue
            
        record = {"Model Variant": run_folder}
        
        # Load Pre-KD (Pruned Baseline)
        pre_eval_path = os.path.join(run_path, "pre_kd_eval.json")
        pre_eval = {}
        if os.path.exists(pre_eval_path):
            with open(pre_eval_path, 'r') as f:
                pre_eval = json.load(f)
                
        # Load Post-KD (Distilled Model)
        post_eval_path = os.path.join(run_path, "post_kd_eval.json")
        post_eval = {}
        if os.path.exists(post_eval_path):
            with open(post_eval_path, 'r') as f:
                post_eval = json.load(f)
            if "post_kd" in post_eval:
                post_eval = post_eval["post_kd"]
                
        # Map values side by side
        for metric in metrics_to_compare:
            pre_val = pre_eval.get(metric, None)
            post_val = post_eval.get(metric, None)
            
            if "perplexity" in metric:
                record[(metric, "Pre-KD")] = f"{pre_val:.2f}" if pre_val is not None else "N/A"
                record[(metric, "Post-KD")] = f"{post_val:.2f}" if post_val is not None else "N/A"
            else:
                record[(metric, "Pre-KD")] = f"{pre_val:.4f}" if pre_val is not None else "N/A"
                record[(metric, "Post-KD")] = f"{post_val:.4f}" if post_val is not None else "N/A"
            
        if len(record) > 1:
            compare_records.append(record)

    if compare_records:
        df_compare = pd.DataFrame(compare_records)
        df_compare.set_index("Model Variant", inplace=True)
        
        # Initialize MultiIndex columns for a grouped tabular design
        df_compare.columns = pd.MultiIndex.from_tuples(df_compare.columns)
        
        print("--- Pre-KD vs Post-KD Distillation Comparison ---")
        display(df_compare)
else:
    print(f"Directory not found: {kd_dir}")